# Episode 3: SFT: fine-tuning a GPT to follow instructions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep003-sft.ipynb)

Our pretrained model learns to write the story an instruction asks for, with a loss on the story alone, and we measure how much better it follows instructions.

[The video](https://youtu.be/WUsVr66Bgq0) · [The episode's page](https://mvgpt.nenuadrian.com/ep003-sft/) · [The docs](https://nenuadrian.github.io/mvGPT/sft/) · [The code at `ep003/end`](https://github.com/nenuadrian/mvGPT/tree/ep003/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep003/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain")

## The data
The stories, and the instructions: TinyStories-Instruct.

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids
!head -n 4 data/instruct_train.txt

In [ ]:
print(open("data/instruct_train.txt").read().split("<|endoftext|>")[4])

## Before: the pretrained model, given an instruction

In [ ]:
!python sample.py checkpoints/pretrain/model.pt "Words: dog, ball, happy. Story:" -n 2

## Fine-tuning
Ten minutes, with the loss on the story alone.

In [ ]:
!python sft.py 10

## After

In [ ]:
!python sample.py checkpoints/sft/model.pt "Words: dog, ball, happy. Story:" -n 2
!python sample.py checkpoints/sft/model.pt "Features: BadEnding. Summary: Tom loses his red kite on a windy day. Story:" -n 1

## Measuring it
How many of the requested words the stories use, how often they end on their own, and the
loss on held-out stories, before and after.

In [ ]:
!python evaluate.py checkpoints/pretrain/model.pt checkpoints/sft/model.pt --out results

## The run in TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

And its logs, read back in Python and plotted with matplotlib:

In [ ]:
import glob

from IPython.display import Image, display

!python plot.py runs/*
for chart in sorted(glob.glob("results/*.png")):
    display(Image(chart))

Next: [episode 4, KV cache from scratch in PyTorch](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep004-kv-cache.ipynb)